<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización de procesos</p>
  </div>

</div>

# **Estructuras de datos aplicadas a la automatización**

## *Listas, tuplas, diccionarios, conjuntos y comprehensions*

En automatización rara vez procesamos un único elemento. Normalmente recibimos:

- colecciones de archivos;
- registros;
- extensiones permitidas;
- reglas de clasificación;
- resultados;
- errores;
- configuraciones.

Por ello, las estructuras de datos de Python son una pieza fundamental.

Seguiremos el patrón:

> **datos reales → inspección → inconsistencia → estructura adecuada → transformación → validación → interpretación**

## Objetivos

Al finalizar deberías poder:

1. Elegir entre lista, tupla, diccionario y conjunto.
2. Recorrer colecciones de forma segura.
3. Modificar y filtrar listas.
4. Representar registros con diccionarios.
5. Construir tablas a partir de listas de diccionarios.
6. Utilizar sets para detectar duplicados y categorías desconocidas.
7. Utilizar comprehensions sin perder legibilidad.
8. Combinar estructuras anidadas.
9. Aplicar reglas de clasificación.
10. Crear un pequeño motor de procesamiento documental.
11. Validar que ningún elemento desaparece durante una automatización.

## 1. Preparación

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import random

import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)

pd.set_option("display.max_columns", 50)

print("Entorno preparado.")

# 2. Listas: una cola de elementos que debemos procesar

Una lista conserva orden y admite elementos repetidos.

Esto la hace apropiada para representar, por ejemplo, los archivos que llegan a una bandeja de entrada.

In [ ]:
archivos = [
    "factura_001.pdf",
    "clientes.xlsx",
    "factura_002.pdf",
    "foto.jpg",
    "ventas.csv",
    "factura_001.pdf",
    "README",
    "informe.PDF",
    "backup.zip"
]

archivos

## 2.1 Inspeccionar antes de transformar

In [ ]:
print("Número de elementos:", len(archivos))
print("Primer archivo:", archivos[0])
print("Último archivo:", archivos[-1])
print("Tres primeros:", archivos[:3])

### Interpretación

El índice permite acceder a posiciones concretas, pero en automatización normalmente recorreremos la colección completa.

## 2.2 Recorrer una lista

In [ ]:
for posicion, archivo in enumerate(
    archivos,
    start=1
):
    print(
        posicion,
        "->",
        archivo
    )

## 2.3 Detectar una primera inconsistencia: duplicados

In [ ]:
conteo = Counter(archivos)

duplicados = {
    archivo: cantidad
    for archivo, cantidad
    in conteo.items()
    if cantidad > 1
}

duplicados

No eliminamos duplicados automáticamente sin entender el proceso.

Dos archivos con el mismo nombre podrían representar:

- un duplicado accidental;
- una nueva versión;
- una colisión de nombres;
- dos entradas que deben conservarse.

Primero detectamos; después decidimos.

## 2.4 Visualizar frecuencias

In [ ]:
frecuencias = pd.Series(
    archivos
).value_counts()

plt.figure(figsize=(10, 5))
plt.bar(
    frecuencias.index,
    frecuencias.values
)
plt.ylabel("Apariciones")
plt.title("Frecuencia de nombres de archivo")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# 3. Modificar listas de forma controlada

In [ ]:
cola = archivos.copy()

cola.append(
    "nuevo_documento.txt"
)

cola.extend([
    "presupuesto.xlsx",
    "imagen.png"
])

print(
    "Elementos después de añadir:",
    len(cola)
)

## 3.1 Error conceptual frecuente: modificar mientras iteramos

In [ ]:
numeros = [1, 2, 3, 4, 5, 6]

ejemplo_problematico = numeros.copy()

for numero in ejemplo_problematico:
    if numero % 2 == 0:
        ejemplo_problematico.remove(
            numero
        )

print(
    "Resultado:",
    ejemplo_problematico
)

### ¿Por qué es peligroso?

Modificar una lista mientras se recorre puede alterar posiciones y producir resultados difíciles de razonar.

Es preferible construir una nueva colección.

## 3.2 Corrección mediante filtrado

In [ ]:
numeros = [1, 2, 3, 4, 5, 6]

impares = [
    numero
    for numero in numeros
    if numero % 2 != 0
]

impares

# 4. Tuplas: datos que conceptualmente no deberían cambiar

In [ ]:
EXTENSIONES_BASE = (
    ".pdf",
    ".xlsx",
    ".csv",
    ".txt"
)

EXTENSIONES_BASE

Una tupla es útil para representar colecciones conceptualmente estables.

No significa seguridad absoluta, pero comunica intención: estos valores forman parte de una definición fija del proceso.

## 4.1 Provocar modificación inválida

In [ ]:
try:
    EXTENSIONES_BASE[0] = ".docx"

except TypeError as error:
    print(
        type(error).__name__,
        "->",
        error
    )

# 5. Diccionarios: reglas y registros

Los diccionarios relacionan una **clave** con un **valor**.

Son especialmente útiles para:

- configuraciones;
- reglas;
- registros;
- resultados;
- metadatos.

In [ ]:
reglas = {
    ".pdf": "PDF",
    ".xlsx": "Excel",
    ".csv": "Datos",
    ".txt": "Texto",
    ".jpg": "Imagenes",
    ".jpeg": "Imagenes",
    ".png": "Imagenes"
}

reglas

## 5.1 Consultar reglas

In [ ]:
print(
    reglas[".pdf"]
)

print(
    reglas.get(
        ".docx",
        "Otros"
    )
)

### `dict.get()`

En automatización suele ser preferible cuando una clave puede no existir.

Permite definir explícitamente un comportamiento de fallback.

## 5.2 Provocar clave desconocida

In [ ]:
try:
    print(
        reglas[".zip"]
    )

except KeyError as error:
    print(
        type(error).__name__,
        "-> clave no configurada:",
        error
    )

## 5.3 Corregir con comportamiento controlado

In [ ]:
categoria_zip = reglas.get(
    ".zip",
    "Revision"
)

print(
    categoria_zip
)

# 6. Diccionarios como registros

In [ ]:
registro = {
    "archivo": "factura_001.pdf",
    "extension": ".pdf",
    "categoria": "PDF",
    "estado": "procesado",
    "tamano_kb": 250
}

registro

## 6.1 Acceder y actualizar

In [ ]:
print(
    registro["archivo"]
)

registro["estado"] = "validado"

registro["requiere_revision"] = False

registro

# 7. Listas de diccionarios: patrón fundamental

In [ ]:
registros = [
    {
        "archivo": "factura_001.pdf",
        "categoria": "PDF",
        "estado": "automatico"
    },
    {
        "archivo": "clientes.xlsx",
        "categoria": "Excel",
        "estado": "automatico"
    },
    {
        "archivo": "backup.zip",
        "categoria": "Revision",
        "estado": "revision"
    }
]

registros

## 7.1 Convertir registros en tabla

In [ ]:
tabla_registros = pd.DataFrame(
    registros
)

tabla_registros

### Interpretación

Este patrón conecta Python básico con el trabajo posterior en Pandas:

```text
lista
  ↓
diccionarios
  ↓
registros estructurados
  ↓
DataFrame
```

# 8. Sets: detectar categorías únicas y duplicados

In [ ]:
extensiones = [
    Path(archivo).suffix.lower()
    if Path(archivo).suffix
    else "sin_extension"
    for archivo in archivos
]

extensiones

In [ ]:
extensiones_unicas = set(
    extensiones
)

extensiones_unicas

Un `set`:

- elimina duplicados;
- no debe utilizarse cuando necesitamos preservar un orden semántico;
- permite operaciones de conjuntos muy útiles para validación.

## 8.1 Detectar extensiones no soportadas

In [ ]:
soportadas = set(
    reglas.keys()
)

observadas = {
    extension
    for extension
    in extensiones
    if extension != "sin_extension"
}

no_soportadas = (
    observadas
    - soportadas
)

no_soportadas

## 8.2 Detectar reglas configuradas que todavía no aparecen

In [ ]:
no_observadas = (
    soportadas
    - observadas
)

no_observadas

### Interpretación

Las operaciones de conjuntos permiten formular validaciones muy expresivas:

```text
observadas - permitidas
```

equivale a:

> ¿qué ha aparecido que no sé procesar?

# 9. Comprehensions

Una comprehension permite construir una colección a partir de otra.

Debe utilizarse cuando mejora la claridad, no simplemente para escribir menos líneas.

## 9.1 List comprehension

In [ ]:
pdfs = [
    archivo
    for archivo in archivos
    if Path(archivo).suffix.lower()
    == ".pdf"
]

pdfs

## 9.2 Dict comprehension

In [ ]:
extension_por_archivo = {
    archivo:
        (
            Path(archivo).suffix.lower()
            or "sin_extension"
        )
    for archivo in archivos
}

extension_por_archivo

## 9.3 Set comprehension

In [ ]:
categorias_observadas = {
    reglas.get(
        Path(archivo).suffix.lower(),
        "Revision"
    )
    for archivo in archivos
}

categorias_observadas

## 9.4 Una comprehension demasiado compleja

In [ ]:
resultado_dificil = [
    (
        archivo,
        reglas.get(
            Path(archivo).suffix.lower(),
            "Sin_extension"
            if not Path(archivo).suffix
            else "Revision"
        ),
        len(archivo)
    )
    for archivo in archivos
]

resultado_dificil[:3]

### Interpretación

Aunque sea válido, empieza a ser difícil de mantener.

Cuando una transformación acumula reglas, conviene moverla a una función.

# 10. De comprehension compleja a función legible

In [ ]:
def analizar_archivo(
    archivo: str
) -> dict:

    ruta = Path(
        archivo
    )

    extension = (
        ruta.suffix.lower()
        or "sin_extension"
    )

    if extension == "sin_extension":
        categoria = "Sin_extension"
        estado = "revision"

    else:
        categoria = reglas.get(
            extension,
            "Revision"
        )

        estado = (
            "revision"
            if categoria == "Revision"
            else "automatico"
        )

    return {
        "archivo": archivo,
        "extension": extension,
        "categoria": categoria,
        "estado": estado,
        "longitud_nombre": len(
            archivo
        )
    }


analizar_archivo(
    "backup.zip"
)

## 10.1 Procesar toda la colección

In [ ]:
resultado = [
    analizar_archivo(
        archivo
    )
    for archivo in archivos
]

resultado[:3]

In [ ]:
df_resultado = pd.DataFrame(
    resultado
)

df_resultado

# 11. Validar que no perdemos elementos

In [ ]:
assert len(
    df_resultado
) == len(
    archivos
)

assert df_resultado[
    "archivo"
].notna().all()

assert df_resultado[
    "estado"
].isin(
    [
        "automatico",
        "revision"
    ]
).all()

print(
    "Validaciones superadas."
)

### Interpretación

Una automatización debe comprobar invariantes.

En este caso:

- cada entrada produce exactamente una salida;
- ninguna salida pierde el nombre;
- todos los estados son conocidos.

# 12. Agrupar resultados con diccionarios

In [ ]:
por_categoria = defaultdict(
    list
)

for registro in resultado:
    por_categoria[
        registro["categoria"]
    ].append(
        registro["archivo"]
    )

dict(
    por_categoria
)

## 12.1 Contar categorías

In [ ]:
conteo_categoria = {
    categoria:
        len(archivos_categoria)
    for categoria, archivos_categoria
    in por_categoria.items()
}

conteo_categoria

## 12.2 Visualizar

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    conteo_categoria.keys(),
    conteo_categoria.values()
)

plt.ylabel(
    "Número de archivos"
)

plt.title(
    "Archivos por categoría"
)

plt.xticks(
    rotation=20
)

plt.tight_layout()
plt.show()

# 13. Estructuras anidadas: configuración de un proceso

In [ ]:
configuracion = {
    "entrada": {
        "carpeta": "entrada",
        "recursivo": False
    },

    "salida": {
        "carpeta": "clasificados",
        "crear_si_no_existe": True
    },

    "reglas": {
        "PDF": [
            ".pdf"
        ],
        "Datos": [
            ".csv",
            ".xlsx"
        ],
        "Imagenes": [
            ".jpg",
            ".jpeg",
            ".png"
        ]
    },

    "seguridad": {
        "sobrescribir": False,
        "enviar_desconocidos_revision": True
    }
}

configuracion

## 13.1 Acceder a configuración anidada

In [ ]:
print(
    configuracion[
        "entrada"
    ][
        "carpeta"
    ]
)

print(
    configuracion[
        "seguridad"
    ][
        "sobrescribir"
    ]
)

# 14. Detectar inconsistencia en una configuración

In [ ]:
configuracion_invalida = {
    "entrada": {
        "carpeta": ""
    },

    "salida": {
        "carpeta": None
    },

    "seguridad": {
        "sobrescribir": "no"
    }
}

configuracion_invalida

## 14.1 Validar después de observar el problema

In [ ]:
def validar_configuracion(
    config: dict
):
    errores = []

    entrada = (
        config
        .get(
            "entrada",
            {}
        )
        .get(
            "carpeta"
        )
    )

    salida = (
        config
        .get(
            "salida",
            {}
        )
        .get(
            "carpeta"
        )
    )

    sobrescribir = (
        config
        .get(
            "seguridad",
            {}
        )
        .get(
            "sobrescribir"
        )
    )

    if not entrada:
        errores.append(
            "Falta carpeta de entrada."
        )

    if not salida:
        errores.append(
            "Falta carpeta de salida."
        )

    if not isinstance(
        sobrescribir,
        bool
    ):
        errores.append(
            "sobrescribir debe ser booleano."
        )

    return errores


validar_configuracion(
    configuracion_invalida
)

## 14.2 Corregir

In [ ]:
configuracion_corregida = {
    "entrada": {
        "carpeta": "entrada"
    },

    "salida": {
        "carpeta": "clasificados"
    },

    "seguridad": {
        "sobrescribir": False
    }
}

errores = validar_configuracion(
    configuracion_corregida
)

print(
    "Errores:",
    errores
)

# 15. Simular un lote más realista

In [ ]:
extensiones_posibles = [
    ".pdf",
    ".xlsx",
    ".csv",
    ".txt",
    ".jpg",
    ".zip",
    ""
]

lote = []

for i in range(
    1,
    101
):
    extension = random.choice(
        extensiones_posibles
    )

    lote.append(
        f"documento_{i:03d}{extension}"
    )

lote[:15]

## 15.1 Procesar el lote

In [ ]:
resultado_lote = pd.DataFrame(
    [
        analizar_archivo(
            archivo
        )
        for archivo
        in lote
    ]
)

resultado_lote.head(
    10
)

## 15.2 Visualizar automático frente a revisión

In [ ]:
estado = (
    resultado_lote[
        "estado"
    ]
    .value_counts()
)

plt.figure(figsize=(7, 5))

plt.bar(
    estado.index,
    estado.values
)

plt.ylabel(
    "Archivos"
)

plt.title(
    "Automatización del lote"
)

plt.tight_layout()
plt.show()

## 15.3 Analizar extensiones problemáticas

In [ ]:
revision = resultado_lote[
    resultado_lote[
        "estado"
    ] == "revision"
]

revision[
    "extension"
].value_counts()

### Interpretación

Ahora podemos cuantificar qué reglas faltan.

Esto es mejor que ampliar reglas a ciegas: utilizamos los propios resultados de la automatización para decidir qué mejorar.

# 16. Duplicados: detectar antes de eliminar

In [ ]:
lote_con_duplicados = (
    lote
    + [
        lote[0],
        lote[0],
        lote[10]
    ]
)

conteos = Counter(
    lote_con_duplicados
)

duplicados = {
    nombre: cantidad
    for nombre, cantidad
    in conteos.items()
    if cantidad > 1
}

duplicados

## 16.1 Estrategia 1: eliminar duplicados sin preservar orden

In [ ]:
unicos_set = list(
    set(
        lote_con_duplicados
    )
)

print(
    len(
        lote_con_duplicados
    ),
    "→",
    len(
        unicos_set
    )
)

## 16.2 Estrategia 2: eliminar duplicados preservando orden

In [ ]:
unicos_ordenados = list(
    dict.fromkeys(
        lote_con_duplicados
    )
)

print(
    unicos_ordenados[:5]
)

print(
    len(
        unicos_ordenados
    )
)

### Interpretación

Ambas soluciones eliminan duplicados, pero no comunican exactamente la misma intención.

En automatización debemos decidir si el orden de llegada forma parte del proceso.

# 17. Mini motor de automatización

In [ ]:
def ejecutar_motor(
    archivos_entrada,
    reglas_clasificacion
):
    registros = []

    for archivo in archivos_entrada:

        ruta = Path(
            archivo
        )

        extension = (
            ruta.suffix.lower()
            or "sin_extension"
        )

        if extension == "sin_extension":
            categoria = "Sin_extension"

        else:
            categoria = (
                reglas_clasificacion
                .get(
                    extension,
                    "Revision"
                )
            )

        registros.append({
            "archivo":
                archivo,

            "extension":
                extension,

            "categoria":
                categoria,

            "estado":
                (
                    "revision"
                    if categoria
                    in {
                        "Revision",
                        "Sin_extension"
                    }
                    else "automatico"
                )
        })

    return registros


salida_motor = ejecutar_motor(
    archivos,
    reglas
)

pd.DataFrame(
    salida_motor
)

# 18. KPIs del motor

In [ ]:
tabla_motor = pd.DataFrame(
    salida_motor
)

kpis = {
    "entradas":
        len(
            archivos
        ),

    "salidas":
        len(
            tabla_motor
        ),

    "categorias":
        tabla_motor[
            "categoria"
        ].nunique(),

    "revision":
        int(
            (
                tabla_motor[
                    "estado"
                ]
                == "revision"
            ).sum()
        )
}

kpis[
    "tasa_revision"
] = (
    kpis[
        "revision"
    ]
    / kpis[
        "entradas"
    ]
)

kpis

## 18.1 Validar invariantes

In [ ]:
assert (
    kpis["entradas"]
    == kpis["salidas"]
)

assert not (
    tabla_motor[
        "archivo"
    ]
    .isna()
    .any()
)

assert set(
    tabla_motor[
        "estado"
    ]
) <= {
    "automatico",
    "revision"
}

print(
    "Motor validado."
)

# 19. ¿Qué estructura elegir?

In [ ]:
comparativa = pd.DataFrame({
    "estructura": [
        "list",
        "tuple",
        "dict",
        "set"
    ],

    "orden": [
        "Sí",
        "Sí",
        "Sí",
        "No usarlo como garantía semántica"
    ],

    "duplicados": [
        "Sí",
        "Sí",
        "Claves únicas",
        "No"
    ],

    "uso_automatizacion": [
        "Cola de archivos",
        "Valores estables",
        "Reglas/configuración/registros",
        "Validación y unicidad"
    ]
})

comparativa

# 20. Mini-práctica guiada — Bandeja de entrada documental

Recibes una colección con al menos 30 nombres de archivo.

Debes construir:

```text
lista de entrada
      ↓
inspección
      ↓
detección de duplicados
      ↓
normalización
      ↓
reglas en diccionario
      ↓
clasificación
      ↓
lista de registros
      ↓
detección con sets
      ↓
DataFrame
      ↓
KPIs
      ↓
gráfico
```

Incluye deliberadamente:

- mayúsculas;
- duplicados;
- extensiones desconocidas;
- archivos sin extensión.

# 21. Ejercicios finales

### Ejercicio 1 — Listas
Crea una lista de 20 documentos y recórrela con `enumerate()`.

### Ejercicio 2 — Duplicados
Introduce duplicados y detecta cuáles son.

### Ejercicio 3 — Tuplas
Define una tupla de extensiones básicas e intenta modificarla.

### Ejercicio 4 — Diccionario
Crea reglas extensión → categoría.

### Ejercicio 5 — KeyError
Provoca una clave inexistente y después corrígelo con `get()`.

### Ejercicio 6 — Registros
Representa un documento mediante un diccionario con cinco campos.

### Ejercicio 7 — Lista de diccionarios
Construye diez registros y conviértelos a DataFrame.

### Ejercicio 8 — Sets
Obtén las extensiones únicas.

### Ejercicio 9 — Diferencia de conjuntos
Detecta extensiones observadas pero no soportadas.

### Ejercicio 10 — Comprehension
Obtén únicamente los PDF mediante list comprehension.

### Ejercicio 11 — Legibilidad
Convierte una comprehension compleja en una función.

### Ejercicio 12 — Configuración
Construye un diccionario anidado de configuración.

### Ejercicio 13 — Inconsistencia
Introduce una configuración incorrecta y muéstrala antes de validarla.

### Ejercicio 14 — Validación
Crea una función que devuelva todos los errores encontrados.

### Ejercicio 15 — Lote
Genera automáticamente 1.000 nombres de archivo.

### Ejercicio 16 — KPIs
Calcula tasa automática y tasa de revisión.

### Ejercicio 17 — Visualización
Representa categorías y estados.

### Ejercicio 18 — Reto
Construye un motor que acepte una lista de archivos y un diccionario de reglas y devuelva registros completos sin perder ninguna entrada.

# 22. Preguntas de reflexión

1. ¿Cuándo utilizarías una lista?
2. ¿Qué intención comunica una tupla?
3. ¿Por qué los diccionarios son adecuados para reglas?
4. ¿Qué aporta `dict.get()`?
5. ¿Qué utilidad tienen los sets?
6. ¿Qué diferencia hay entre eliminar duplicados con `set` y `dict.fromkeys`?
7. ¿Cuándo una comprehension deja de ser recomendable?
8. ¿Por qué una lista de diccionarios es un patrón tan útil?
9. ¿Qué es una estructura anidada?
10. ¿Por qué debemos validar configuración?
11. ¿Qué invariantes comprobarías en un procesamiento por lotes?
12. ¿Por qué no debemos eliminar duplicados antes de inspeccionarlos?
13. ¿Cómo ayudan estas estructuras al futuro Organizador Inteligente de Documentos?

# 23. Cierre

En este laboratorio hemos construido las estructuras internas que necesitarán nuestras automatizaciones.

El siguiente paso será encapsular la lógica mediante:

```text
funciones
   ↓
parámetros
   ↓
valores de retorno
   ↓
módulos
   ↓
separación de responsabilidades
   ↓
código reutilizable
```

Eso nos permitirá abandonar progresivamente el código monolítico y preparar automatizaciones mantenibles.